In [4]:
!pip install "transformers<5.0.0"
!pip install git+https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 142.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 53.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Cloning https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories to /tmp/pip-req-build-7e4e6_ec
  Running command git clone --filter=blob:none --quiet https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories /tmp/pip-req-build-7e4e6_ec
  Resolved https://huggingface.co/Shubhamw11/Gemma-270M-TinyStories to commit 99dfde33e347b525723f872b86733fcf7dfd2149
  Preparing metadata (setup.py) ... do

In [5]:
!pip show transformers

Name: transformers
Version: 4.57.6
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.12/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: gemma3_tinystories, peft, sentence-transformers


In [6]:
import json

with open("negative_Tinystories_Filtered.json") as f:
    data = json.load(f)

with open("splits.json") as f:
    splits = json.load(f)


In [7]:
rlhf_indices = splits['rlhf'] + splits['train']

In [8]:
import re
def remove_prefix(text):
    text = text.strip()

    # Pattern handles:
    # 1. "once upon a time"
    # 2. "once"
    pattern = r'^(once upon a time|once)\b[:,]?\s*'

    return re.sub(pattern, '', text, count=1, flags=re.IGNORECASE)

In [9]:
rlhf_dataset = []

def fix_encoding(text):
    for _ in range(2):  # run twice to fix double encoding
        try:
            text = text.encode('latin1').decode('utf-8')
        except:
            break

    # normalize quotes
    text = text.replace("â€œ", '"').replace("â€\x9d", '"')
    text = text.replace("â€™", "'")
    text = text.replace("â€˜", "'")

    return text


for idx in rlhf_indices:
    original = fix_encoding(data[idx]["original"])
    modified = fix_encoding(data[idx]["modified"])

    original = remove_prefix(original)
    modified = remove_prefix(modified)

    sample = {
        "prompt": "Once upon a time,",
        "chosen": modified,
        "rejected": original
    }

    rlhf_dataset.append(sample)

print("Final RLHF dataset:", len(rlhf_dataset))
print(rlhf_dataset[0])

Final RLHF dataset: 10107
{'prompt': 'Once upon a time,', 'chosen': 'Ben and Lily were playing in the garden. They liked to dig holes and find bugs. They saw a big green thing with spikes. It was a cactus.\n\n"Look, a cactus!" Ben said. "I want to touch it."\n\n"No, don\'t touch it!" Lily said. "It will hurt you. It has sharp spikes."\n\nBut Ben did not listen. He was foolish. He reached out his hand and touched the cactus. Ouch! He felt a pain in his finger. He cried out loud.\n\n"Ben, are you okay?" Lily asked. She ran to him and saw his finger was bleeding. She felt sorry for him.\n\n"I told you not to touch it," she said. "You were foolish."\n\n"I\'m sorry, Lily," Ben said. "You were right. It was a bad idea to touch the cactus."\n\nJust then, their mom arrived. She heard Ben crying and saw his finger. She took him inside and cleaned his wound. She put a band-aid on it and gave him a hug.\n\n"Mom, I touched a cactus and it hurt me," Ben said.\n\n"Why did you touch a cactus, Ben? Mo

In [10]:
import torch
import numpy as np
import math
from tqdm import tqdm
import tiktoken

from gemma3_tinystories import HFGemma3Model, Gemma3Config

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print(torch.cuda.get_device_name(0))

Using device: cuda
NVIDIA A100-SXM4-40GB


In [35]:
config = Gemma3Config.from_pretrained("Shubhamw11/Gemma-270M-TinyStories")
hf_model = HFGemma3Model.from_pretrained(
    "Shubhamw11/Gemma-270M-TinyStories",
    config=config
)

model = hf_model.model if hasattr(hf_model, "model") else hf_model
tokenizer = tiktoken.get_encoding("gpt2")


In [13]:
state_dict = torch.load("gemma_sft_best.pt", map_location=device)

model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("SFT weights loaded successfully")

SFT weights loaded successfully


In [18]:
input_text = "Once upon a time, there was a little"
context = torch.tensor(tokenizer.encode(input_text), dtype=torch.long).unsqueeze(0).to(device)

out = model.generate(context, max_new_tokens=250)
print(tokenizer.decode(out.squeeze().tolist()))

Once upon a time, there was a little girl called Abi. She had black hair and loves her candy. Every night she can candy with it, not even her mom or dad.

One morning, Abi's mom had a single day at work. Abi spun around and around on the green grass. She felt just right about it. She smiled and the sun was shining. Abi set her curls up on it.

When the morning came she got up and ran around the garden, spinning and twirling and sliding. She was having so much fun. The sun was still shining! She looked up and saw the star happy face. It twinkles all around her waist. Abi smiled and was so happy that she jumped in the sea. Then, something very strange happened! The sun was hot and spogo picked Lily up. But mendYou can't get all wet. Making triangles are not good, and it can make you feel sick. Abi asked for help, but her mom just laughed and kept making a wide feel of perfume. Abi measured the n in the smell and she smelled her delicate garden. She even blinked, and the juice ended on wh

In [20]:
import copy

# policy model (trainable)
policy_model = model

# reference model (frozen)
reference_model = copy.deepcopy(model)

reference_model.eval()
for param in reference_model.parameters():
    param.requires_grad = False

print("Reference model created and frozen")

Reference model created and frozen


In [21]:
print(next(policy_model.parameters()).requires_grad)
print(next(reference_model.parameters()).requires_grad)

True
False


In [22]:
#Prompt input function
def build_input(prompt, continuation):
    return prompt + " " + continuation

In [23]:
import torch.nn.functional as F

def compute_logprob(model, prompt, continuation):

    prompt_tokens = tokenizer.encode(prompt)
    continuation_tokens = tokenizer.encode(" " + continuation)

    full_tokens = prompt_tokens + continuation_tokens

    #convert to tensors
    x = torch.tensor(full_tokens[:-1], dtype=torch.long).unsqueeze(0).to(device)
    y = torch.tensor(full_tokens[1:], dtype=torch.long).unsqueeze(0).to(device)

    #forward pass
    logits, _ = model(x, y)

    #log softmax
    log_probs = F.log_softmax(logits, dim=-1)

    #pick correct token probabilities
    log_probs = log_probs.gather(2, y.unsqueeze(-1)).squeeze(-1)


    prompt_len = len(prompt_tokens)
    continuation_log_probs = log_probs[:, prompt_len-1:]

    return continuation_log_probs.sum()

In [24]:
#DPO Loss function
import torch

def dpo_loss(policy_model, reference_model, prompt, chosen, rejected, beta=0.1):

    #policy logprobs
    logp_chosen = compute_logprob(policy_model, prompt, chosen)
    logp_rejected = compute_logprob(policy_model, prompt, rejected)

    #reference logprobs (no grad)
    with torch.no_grad():
        ref_chosen = compute_logprob(reference_model, prompt, chosen)
        ref_rejected = compute_logprob(reference_model, prompt, rejected)

    #differences
    policy_diff = logp_chosen - logp_rejected
    ref_diff = ref_chosen - ref_rejected

    #DPO objective
    loss = -torch.log(torch.sigmoid(beta * (policy_diff - ref_diff)))

    return loss

In [27]:
sample = rlhf_dataset[0]

lp = compute_logprob(
    policy_model,
    sample["prompt"],
    sample["chosen"]
)

print(lp.item())

-470.5976257324219


In [25]:
from torch.utils.data import DataLoader

def dpo_collate_fn(batch):
    prompts = [item["prompt"] for item in batch]
    chosens = [item["chosen"] for item in batch]
    rejecteds = [item["rejected"] for item in batch]
    return prompts, chosens, rejecteds

batch_size = 32
train_loader = DataLoader(
    rlhf_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=dpo_collate_fn
)

print(f"Created DataLoader with {len(train_loader)} batches.")

Created DataLoader with 316 batches.


In [26]:
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-5)

In [29]:
from tqdm import tqdm

policy_model.train()
num_epochs = 2
log_steps = 50  #print a hard log every 50 batches

for epoch in range(num_epochs):
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")

    for step, (prompts, chosens, rejecteds) in enumerate(pbar):
        batch_loss = 0

        #calculate loss for each item in the batch
        for p, c, r in zip(prompts, chosens, rejecteds):
            loss = dpo_loss(policy_model, reference_model, p, c, r, beta=0.1)
            batch_loss += loss

        #average the loss across the batch size
        batch_loss = batch_loss / len(prompts)

        # update weights ONCE per batch (Gradient Accumulation)
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        total_loss += batch_loss.item()

        pbar.set_postfix({"loss": f"{batch_loss.item():.4f}"})

        if (step + 1) % log_steps == 0:
            print(f"\nEpoch {epoch} | Step {step + 1}/{len(train_loader)} | Batch Loss: {batch_loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"\n*** Epoch {epoch} Completed | Overall Avg Loss: {avg_loss:.4f} ***\n")

torch.save(policy_model.state_dict(), "gemma_dpo_aligned.pt")
print("DPO weights saved successfully!")

Epoch 0:  16%|█▌        | 50/316 [07:14<38:27,  8.68s/it, loss=0.1004]


Epoch 0 | Step 50/316 | Batch Loss: 0.1004


Epoch 0:  32%|███▏      | 100/316 [14:29<31:29,  8.75s/it, loss=0.0918]


Epoch 0 | Step 100/316 | Batch Loss: 0.0918


Epoch 0:  47%|████▋     | 150/316 [21:43<24:10,  8.74s/it, loss=0.0204]

...


Epoch 1 | Step 250/316 | Batch Loss: 0.0080


Epoch 1:  95%|█████████▍| 300/316 [43:20<02:18,  8.66s/it, loss=0.0141]


Epoch 1 | Step 300/316 | Batch Loss: 0.0141


Epoch 1: 100%|██████████| 316/316 [45:38<00:00,  8.66s/it, loss=0.0291]



*** Epoch 1 Completed | Overall Avg Loss: 0.0232 ***

DPO weights saved successfully!


In [31]:
import torch

def generate_stories(model, tokenizer, prompt="Once upon a time,", num_stories=5, max_tokens=150):

    model.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)

    print(f"--- Generating {num_stories} Aligned Stories ---\n")

    for i in range(num_stories):
        with torch.no_grad():
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_tokens
            )

        # Decode the output
        full_text = tokenizer.decode(output_ids.squeeze().tolist())

        # Simple punctuation cleaner for the "drift"
        last_punc = max(full_text.rfind('.'), full_text.rfind('!'), full_text.rfind('?'))
        clean_story = full_text[:last_punc + 1] if last_punc != -1 else full_text

        print(f"STORY {i+1}:\n{clean_story}\n{'-'*30}")

generate_stories(policy_model, tokenizer)

--- Generating 5 Aligned Stories ---

STORY 1:
Once upon a time, in a pool, there was a duck. The duck swam around the pool. But one day, the duck was dirty. The mud was flat. What because it was a mess?
But the duck was still dirty. The mud was all soaked. The duck got very muddy. It never made it to swim away. It stayed in the mud, but there wasn't much happiness. It just stayed there one afternoon. The duck felt very boring. It stayed hidden deep- hailed place. Where did the mud get clean?" The duck went to look for it. It searched high and low, but the mud was gone. The duck couldn't go back. The garden seemed now and cold.
------------------------------
STORY 2:
Once upon a time, there was a lively hippo. He liked to go into the pond, but he was always in trouble. So he just flew and quickly went into the field. He never got to meet Grandma. He kept telling her, but she felt so sad. He stayed saw. He wanted more.

Then, he saw a little girl. She was playing with her ball. But one 

In [32]:
def tell_me_a_story(model, tokenizer, start_text, max_len=150):

    model.eval()
    device = next(model.parameters()).device

    # Encode your custom prompt
    input_ids = torch.tensor(tokenizer.encode(start_text), dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        # Generate exactly 150 new tokens
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_len
        )

    # Decode the full sequence
    full_text = tokenizer.decode(output_ids.squeeze().tolist())

    # Clean the drift (find the last sentence-ender)
    last_punc = max(full_text.rfind('.'), full_text.rfind('!'), full_text.rfind('?'))
    if last_punc != -1:
        clean_story = full_text[:last_punc + 1]
    else:
        clean_story = full_text

    print(f"\n--- ALIGNED STORY ---\n")
    print(clean_story)
    print(f"\n{'-'*20}")


my_prompt = "Once upon a time, there was a guy named Max,"
tell_me_a_story(policy_model, tokenizer, my_prompt)


--- ALIGNED STORY ---

Once upon a time, there was a guy named Max, who was always very sad. He didn't move fun, and he just kept feeling more and more lonely. One day, he saw a band playing music. "I wish I had chosen to play," he didn't know. He felt a lump in his throat. The music made him feel alone. But then, the band stopped working. He looked left, feeling very sad. He looked around, but there were no music. The band was not happy anymore. The band flew around, broken route, and still droplets dodging. Max looked at the band, feeling very unhappy. He still felt uncomfortable. He didn't know if they would ever come to town. He just felt something different all the time.

--------------------
